In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.transforms.v2 as v2

In [2]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [3]:
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

training_data = datasets.MNIST(
    root='data',
    download=True,
    train=True,
    transform=transform
)

testing_data = datasets.MNIST(
    root='data',
    download=True,
    train=False,
    transform=transform
)

train_loader = DataLoader(dataset=training_data, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=testing_data, batch_size=1000, shuffle=False)

In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.stack = nn.Sequential(
            nn.Linear(28*28, 32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logit = self.stack(x)
        return logit

In [19]:
model = NeuralNetwork().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [15]:
def train(model, dataloader, loss_fn, optimizer, epochs=20):
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0

        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            pred = model(images)
            loss = loss_fn(pred, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(dataloader):.4f}")

def test(model, dataloader):
    model.eval()
    correct = 0
    size = len(dataloader.dataset)

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            pred = model(images)
            correct += (pred.argmax(1) == labels).sum().item()
    print(f'Accuracy on test set: {correct / size:2%}') 

In [20]:
train(model, train_loader, loss_fn, optimizer)

Epoch 1/20 - Loss: 0.4537
Epoch 2/20 - Loss: 0.2536
Epoch 3/20 - Loss: 0.2065
Epoch 4/20 - Loss: 0.1753
Epoch 5/20 - Loss: 0.1529
Epoch 6/20 - Loss: 0.1362
Epoch 7/20 - Loss: 0.1235
Epoch 8/20 - Loss: 0.1125
Epoch 9/20 - Loss: 0.1033
Epoch 10/20 - Loss: 0.0959
Epoch 11/20 - Loss: 0.0897
Epoch 12/20 - Loss: 0.0830
Epoch 13/20 - Loss: 0.0775
Epoch 14/20 - Loss: 0.0730
Epoch 15/20 - Loss: 0.0687
Epoch 16/20 - Loss: 0.0651
Epoch 17/20 - Loss: 0.0618
Epoch 18/20 - Loss: 0.0587
Epoch 19/20 - Loss: 0.0557
Epoch 20/20 - Loss: 0.0521


In [21]:
test(model, test_loader)

Accuracy on test set: 96.740000%
